# Task 5: Custom BatchNorm & LayerNorm forward/backward (NumPy)

In [1]:
import numpy as np


In [2]:
def batchnorm_forward(x, gamma, beta, eps=1e-5):
    mu = x.mean(axis=0)
    var = x.var(axis=0)
    xhat = (x - mu) / np.sqrt(var + eps)
    out = gamma * xhat + beta
    cache = (x, xhat, mu, var, gamma, eps)
    return out, cache

def batchnorm_backward(dout, cache):
    x, xhat, mu, var, gamma, eps = cache
    N = x.shape[0]
    std_inv = 1.0 / np.sqrt(var + eps)
    dgamma = np.sum(dout * xhat, axis=0)
    dbeta = np.sum(dout, axis=0)
    dxhat = dout * gamma
    dvar = np.sum(dxhat * (x - mu) * -0.5 * std_inv**3, axis=0)
    dmu = np.sum(dxhat * -std_inv, axis=0) + dvar * np.mean(-2*(x-mu), axis=0)
    dx = dxhat*std_inv + dvar*2*(x-mu)/N + dmu/N
    return dx, dgamma, dbeta


In [3]:
def layernorm_forward(x, gamma, beta, eps=1e-5):
    mu = x.mean(axis=1, keepdims=True)
    var = x.var(axis=1, keepdims=True)
    xhat = (x - mu) / np.sqrt(var + eps)
    out = gamma * xhat + beta
    cache = (x, xhat, mu, var, gamma, eps)
    return out, cache

def layernorm_backward(dout, cache):
    x, xhat, mu, var, gamma, eps = cache
    D = x.shape[1]
    std_inv = 1.0 / np.sqrt(var + eps)
    dgamma = np.sum(dout * xhat, axis=0)
    dbeta = np.sum(dout, axis=0)
    dxhat = dout * gamma
    dvar = np.sum(dxhat * (x - mu) * -0.5 * std_inv**3, axis=1, keepdims=True)
    dmu = np.sum(dxhat * -std_inv, axis=1, keepdims=True) + dvar * np.mean(-2*(x-mu), axis=1, keepdims=True)
    dx = dxhat*std_inv + dvar*2*(x-mu)/D + dmu/D
    return dx, dgamma, dbeta


In [4]:
# quick numeric gradient check on batchnorm
np.random.seed(1)
x = np.random.randn(5,4)
gamma = np.ones(4)
beta = np.zeros(4)
out, cache = batchnorm_forward(x, gamma, beta)
dout = np.random.randn(*out.shape)
dx, dgamma, dbeta = batchnorm_backward(dout, cache)

eps = 1e-5
num_dx = np.zeros_like(x)
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        xp = x.copy(); xp[i,j] += eps
        op, _ = batchnorm_forward(xp, gamma, beta)
        xm = x.copy(); xm[i,j] -= eps
        om, _ = batchnorm_forward(xm, gamma, beta)
        num_dx[i,j] = np.sum((op-om)/(2*eps) * dout)

print("max grad diff:", np.abs(num_dx - dx).max())


max grad diff: 1.8799739542885163e-11
